
# rsql 

Registry API can be used with powerful rsql query capabilities ([github.com/jirutka/rsql-parser](https://github.com/jirutka/rsql-parser)).

The most popular parameters used in combination with RSQL are:

- `where`: essential part of the query, to specify the *what*
- `with`: selecting the parts of the data model to return, e.g. contacts or events
- `offset`: skips the first n elements that are returned
- `hits`: restricts the returned entries to n elements
- `sorts`: sorts the returned elements according

For better readability in this document the concatenated strings of the requests are separated into logical entities, such as

```python

basicLink + endPoint + "?with=facetOfChoice1" + "&with=facetOfChoice2" +"&where=something.nested.nested==querriesPhrase"

```


In [1]:
import requests
import json

## sort of base url, ubiquitously in use
regUrl = "https://registry.o2a-data.de/rest/v2/"

## it is capped to 3 results (just for simplicity)
testlink1 = regUrl + "items"  + "?hits=3"

test1 = json.loads(requests.get(testlink1).content)

test1


{'offset': 0,
 'hits': 3,
 'totalHits': 9352,
 'records': [{'@uuid': '12bd8519-ee5f-4d5e-86c6-7175e8dab660',
   'id': 22,
   'created': '2016-01-11T00:00:00',
   'lastModified': '2024-01-18T07:34:31.561',
   'code': 'vessel:polarstern:weather',
   'shortName': 'Weather',
   'longName': 'Weather Station',
   'description': 'The weather station at Polarstern consists of several devices which measure standard parameters such as temperature, wind, humidity, barometric pressure, radiation and precipitation. Satellite images are received with a dedicated antenna which, together with the data, are used for forecast purposes. The station is operated by a weather technician-observer from the German Weather Service (DWD) who is responsible for the routine 3-hourly synoptic observations and the daily upper air soundings.\n',
   'model': 'AWI',
   'manufacturer': 'AWI',
   'serialNumber': '',
   'inventoryNumber': '',
   'citation': '10013/sensor.14ff286a-af24-4be9-b751-6624db07ddd0',
   'parentId


## structure of returned json

First test of our function and its returned content. As can be seen on top level of the json output there are

- `offset`: integer, we set none, thus the returned items are starting from 0
- `hits`: integer, we capped it to three hits only
- `totalHits`: integer, how many hits are available
- `records`: list, that is the content we want
- `duration`: integer, time in ms how long the call needed to return

Each key in the json can be used to query for.

REGISTRY fields (keys) that are not filled with valid content (values) are omitted in the latest data model. That means they are not returned, if they are not set. This is the latest behavior in REGSITRY. For historical reasons some older items have keys, such as `inventoryNumber` in this example, that are filled with an empty value. Newly created items without proper content would not return the key `inventoryNumber`.

Exemplary output of the first entry of the retrieved list looks like: 

In [2]:

test1['records'][0]

{'@uuid': '12bd8519-ee5f-4d5e-86c6-7175e8dab660',
 'id': 22,
 'created': '2016-01-11T00:00:00',
 'lastModified': '2024-01-18T07:34:31.561',
 'code': 'vessel:polarstern:weather',
 'shortName': 'Weather',
 'longName': 'Weather Station',
 'description': 'The weather station at Polarstern consists of several devices which measure standard parameters such as temperature, wind, humidity, barometric pressure, radiation and precipitation. Satellite images are received with a dedicated antenna which, together with the data, are used for forecast purposes. The station is operated by a weather technician-observer from the German Weather Service (DWD) who is responsible for the routine 3-hourly synoptic observations and the daily upper air soundings.\n',
 'model': 'AWI',
 'manufacturer': 'AWI',
 'serialNumber': '',
 'inventoryNumber': '',
 'citation': '10013/sensor.14ff286a-af24-4be9-b751-6624db07ddd0',
 'parentId': 90,
 'status': {'@uuid': 'a080ff68-30b2-450a-9a44-06ecd244c0b7',
  'id': 1,
  'las


For reduction of complexity we create a small function to work with in this example:


In [3]:

def getContent(link):
    x = requests.get(link + "&hits=3")
    if x.status_code == 200:
        y = json.loads(x.content)
        return(y)
    else:
        raise Exception(f'error, {x.status_code}')


## example calls using rsql

With the base url (`regUrl = "https://registry.o2a-data.de/rest/v2/"`) and the annex `items` all items are retrieved. By appending `?where=id==4079` it is possible to filter for certain items by their `id` or `uuid`.


In [4]:

testlink2 = regUrl + "items" + "?where=id==93"
testlink3 = regUrl + "items" + "?where=uuid==bcbc0558-ca33-4c52-bcf0-019a98eb51bd"

a=getContent(testlink2)
b=getContent(testlink3)

print(a)
print("-----------------")
print(b)


{'offset': 0, 'hits': 1, 'totalHits': 1, 'records': [{'@uuid': 'bcbc0558-ca33-4c52-bcf0-019a98eb51bd', 'id': 93, 'lastModified': '2023-07-05T10:39:11.553', 'code': 'vessel:heincke', 'shortName': 'Heincke', 'longName': 'Heincke', 'description': '', 'model': '', 'manufacturer': 'Detlef Hegemann, Rolandwerft GmbH - CoKG, Berne', 'serialNumber': '144', 'inventoryNumber': '', 'citation': '10013/sensor.8f21b4c1-d558-4704-a981-29efbccfbe60', 'status': {'@uuid': 'a080ff68-30b2-450a-9a44-06ecd244c0b7', 'id': 1, 'lastModified': '2020-11-18T09:14:53.011', 'name': 'public', 'meta': 'Item is completely described and operational.'}, 'type': {'@uuid': '32f59e33-05a9-4834-a62e-06c6c34f3ef6', 'id': 236, 'lastModified': '2020-11-18T09:48:51.086', 'generalName': 'vessel', 'systemName': 'vessel', 'description': 'A platform of any size operating on the surface of the water column in unpredictable locations that is specifically equipped, manned and operated for scientific, usually oceanographic, research', 


The same applies for the `code` (urn), `shortName`, `serialNumber`, etc.


In [5]:

testlink4 = regUrl + "items" + "?where=code==vessel:heincke"
a=getContent(testlink4)

testlink5 = regUrl + "items" + "?where=shortName==heincke"
b=getContent(testlink5)

testlink6 = regUrl + "items" + "?where=serialNumber==144"
c=getContent(testlink6)

print(a)
print("-----------------")
print(b)
print("-----------------")
print(c)



{'offset': 0, 'hits': 1, 'totalHits': 1, 'records': [{'@uuid': 'bcbc0558-ca33-4c52-bcf0-019a98eb51bd', 'id': 93, 'lastModified': '2023-07-05T10:39:11.553', 'code': 'vessel:heincke', 'shortName': 'Heincke', 'longName': 'Heincke', 'description': '', 'model': '', 'manufacturer': 'Detlef Hegemann, Rolandwerft GmbH - CoKG, Berne', 'serialNumber': '144', 'inventoryNumber': '', 'citation': '10013/sensor.8f21b4c1-d558-4704-a981-29efbccfbe60', 'status': {'@uuid': 'a080ff68-30b2-450a-9a44-06ecd244c0b7', 'id': 1, 'lastModified': '2020-11-18T09:14:53.011', 'name': 'public', 'meta': 'Item is completely described and operational.'}, 'type': {'@uuid': '32f59e33-05a9-4834-a62e-06c6c34f3ef6', 'id': 236, 'lastModified': '2020-11-18T09:48:51.086', 'generalName': 'vessel', 'systemName': 'vessel', 'description': 'A platform of any size operating on the surface of the water column in unpredictable locations that is specifically equipped, manned and operated for scientific, usually oceanographic, research', 


As can be seen the `serialNumber` in this case is not suitable as a unique identifier, because more than one entry is returned.

The nested structure of the json can be entered by a separating full stop (`.`). For example the `type` object offers several keys, such as `systemName`. To filter for all (in this case it is restricted through the `hits=3` statement in the function) items of a certain type the request is as follows:


In [6]:

testlink7 = regUrl + "items" + "?where=type.systemName==underwater_acoustic_recorder"
c=getContent(testlink7)
c

{'offset': 0,
 'hits': 3,
 'totalHits': 82,
 'records': [{'@uuid': 'aad18a3d-f769-4876-8dca-fc1eed1dcef3',
   'id': 257,
   'created': '2016-03-24T00:00:00',
   'lastModified': '2024-05-15T11:02:40.456',
   'code': 'mooring:awi229-14:sv1060',
   'shortName': 'SV1060',
   'longName': 'SonoVault SV1060',
   'description': 'Passive acoustic underwater recorder',
   'model': 'SonoVault',
   'manufacturer': 'Develogic GmbH, Hamburg, Germany ',
   'serialNumber': '1060',
   'inventoryNumber': '71242',
   'citation': '10013/sensor.420cdb2c-d9cf-441d-8c09-5394f67b771a',
   'parentId': 11355,
   'status': {'@uuid': 'a080ff68-30b2-450a-9a44-06ecd244c0b7',
    'id': 1,
    'lastModified': '2020-11-18T09:14:53.011',
    'name': 'public',
    'meta': 'Item is completely described and operational.'},
   'type': {'@uuid': '31fb67fc-0dc6-4f3c-a275-be47cf8dbbb9',
    'id': 259,
    'lastModified': '2020-11-18T09:16:20.753',
    'generalName': 'underwater acoustic recorder',
    'systemName': 'underwate


The parameter `with` allows to load certain facets of an item together with the item request. For example `?with=parameters` it is possible to retriever the item that has a certain parameter available. The constraint is knowledge about the parameter id or uuid. It might work with the parameter shortname as well, but only if it is truely (globally) unique. If not, there are multiple items returned. 


In [7]:
testlink8 = regUrl + "items" + "?with=contacts&where=id==10290"
a=getContent(testlink8)
print(a)

print('----------------')

testlink9 = regUrl + "items" + "?with=contacts&where=parameters.shortName==tur"
b=getContent(testlink9)
print(b)



{'offset': 0, 'hits': 1, 'totalHits': 1, 'records': [{'@uuid': 'd9c7d04d-4b73-4416-8d3c-1d7f72f8a593', 'id': 10290, 'created': '2024-10-15T09:45:17.123', 'lastModified': '2024-10-15T09:45:17.124', 'code': 'vessel:emb:iow_mbes_sonic2024_101600', 'shortName': 'IOW_MBES_sonic2024_101600', 'longName': 'IOW SONIC 2024 Multibeam Echosounder 101600', 'description': 'r2Sonic Sonic 2024 is a high-resolution seafloor mapping system with beam width down to 0.3° x 0.6° at 700kHz and up to 1° x 2° at 200kHz. Maximum coverage is 10° to 160°, sounding depth up to 400m+ and max. ping rate up to 60Hz.\nWater column data can be stored.', 'model': 'R2Sonic SONIC 2024 Wideband multibeam echosounder', 'manufacturer': 'R2Sonic, LLC', 'serialNumber': '101600', 'inventoryNumber': '', 'citation': '10013/sensor.bf81a24e-38c8-4245-8c75-d0f301491a3f', 'parentId': 9140, 'status': {'@uuid': 'a080ff68-30b2-450a-9a44-06ecd244c0b7', 'id': 1, 'lastModified': '2020-11-18T09:14:53.011', 'name': 'public', 'meta': 'Item is

{'offset': 0, 'hits': 3, 'totalHits': 4, 'records': [{'@uuid': '9a7f6fff-29bc-4b43-9dd4-e7440ff94687', 'id': 61, 'created': '2016-02-15T00:00:00', 'lastModified': '2024-11-15T09:00:54.495', 'code': 'vessel:polarstern:fb_ps', 'shortName': 'FB_PS', 'longName': 'Ferrybox', 'description': 'The FerryBox is an automatic, flow-through system to measure water parameters (e.g. temperature, conductivity, sound velocity, pH, fluorescence) continously and unattended.', 'model': 'Ferrybox COSYNA', 'manufacturer': '4H Jena', 'serialNumber': 'PS', 'inventoryNumber': '', 'citation': '10013/sensor.8590c648-6e2a-4c82-92a6-334ca6533ab0', 'parentId': 90, 'status': {'@uuid': '8fa8fd50-a2bf-49ea-991d-03b85cee1b7f', 'id': 4, 'lastModified': '2020-11-18T09:18:08.688', 'name': 'decom', 'meta': 'Item is retired and not used anymore.'}, 'type': {'@uuid': 'a3083189-c77c-47f4-8ef5-02b72deb5bcb', 'id': 199, 'lastModified': '2022-09-14T10:12:33.445', 'generalName': 'small scale facility', 'systemName': 'small_scale_


In this case all items that have a parameter with the unit [NTU]('https://en.wikipedia.org/wiki/Turbidity') are returned along with its resources.



In [8]:
testlink10 = regUrl + "items" + "?with=resources" + "&with=parameters" +"&where=parameters.unit.code==NTU"
a=getContent(testlink10)
for i in a['records']:
    print(i)
    print('---')


{'@uuid': '9a7f6fff-29bc-4b43-9dd4-e7440ff94687', 'id': 61, 'created': '2016-02-15T00:00:00', 'lastModified': '2024-11-15T09:00:54.495', 'code': 'vessel:polarstern:fb_ps', 'shortName': 'FB_PS', 'longName': 'Ferrybox', 'description': 'The FerryBox is an automatic, flow-through system to measure water parameters (e.g. temperature, conductivity, sound velocity, pH, fluorescence) continously and unattended.', 'model': 'Ferrybox COSYNA', 'manufacturer': '4H Jena', 'serialNumber': 'PS', 'inventoryNumber': '', 'citation': '10013/sensor.8590c648-6e2a-4c82-92a6-334ca6533ab0', 'parentId': 90, 'status': {'@uuid': '8fa8fd50-a2bf-49ea-991d-03b85cee1b7f', 'id': 4, 'lastModified': '2020-11-18T09:18:08.688', 'name': 'decom', 'meta': 'Item is retired and not used anymore.'}, 'type': {'@uuid': 'a3083189-c77c-47f4-8ef5-02b72deb5bcb', 'id': 199, 'lastModified': '2022-09-14T10:12:33.445', 'generalName': 'small scale facility', 'systemName': 'small_scale_facility', 'description': '', 'vocabulary': 'AWI', 'v


The following gets all items with the long name `temperature`. Remark: case sensitivity does not apply.



In [9]:

testlink11 = regUrl + "items/" + "?with=parameters" + "&where=parameters.name==temperature"
a=getContent(testlink11)
a
#

{'offset': 0,
 'hits': 3,
 'totalHits': 278,
 'records': [{'@uuid': '76780fec-3d1f-4946-bc67-96d757411f0e',
   'id': 92,
   'created': '2016-01-04T00:00:00',
   'lastModified': '2021-01-13T08:49:50.613',
   'code': 'vehicle:sg127',
   'shortName': 'SG127',
   'longName': 'Seaglider SN 127',
   'description': 'Seaglider is an autonomous underwater vehicle AUV or underwater glider developed for continuous, long term measurement of oceanographic parameters. Rather than an electrically driven propeller, the vehicle uses small changes in buoyancy and wings to achieve forward motion. The systems pitch and roll are controlled using adjustable ballast the vehicle battery.',
   'model': 'SFC',
   'manufacturer': 'Seaglider Fabrication Center',
   'serialNumber': '127',
   'inventoryNumber': '',
   'citation': '10013/sensor.38199089-e4a3-448f-a299-a3d11619509e',
   'status': {'@uuid': '380e59eb-5f70-4d6f-9f79-c212c5141e20',
    'id': 2,
    'lastModified': '2020-11-18T09:16:20.751',
    'name': 


This retrieves all events of a certain type for an item (id 4044).
**Remark:** a different than the above API endpoints is used here.

In [10]:

testlink12 = regUrl + "items/" + str(4044) + "/events" + "?where=type.systemName==information"
print(testlink12)
a=getContent(testlink12)
a


https://registry.o2a-data.de/rest/v2/items/4044/events?where=type.systemName==information


{'offset': 0,
 'hits': 1,
 'totalHits': 1,
 'records': [{'@uuid': '5cf69b3b-d04d-4c5f-b415-ea783cdafc37',
   'id': 45222,
   'startDate': '2020-10-19T02:00:00',
   'endDate': '2020-10-19T17:00:00',
   'label': 'MOSES',
   'description': 'Aufbau auf Radarturm',
   'type': {'@uuid': 'd865bf37-04c3-44f5-a474-0458dc68f915',
    'id': 50,
    'lastModified': '2020-11-18T09:16:28.347',
    'generalName': 'Information',
    'systemName': 'Information',
    'description': 'Describes issues associated with this item relevant for its operation.  Includes failure and maintenance information.  ',
    'vocabulary': 'AWI',
    'vocableValue': 'Information',
    'vocableGroup': {'@uuid': '2ea05127-95fd-4f2d-aa68-82ea0283aaaf',
     'id': 8,
     'name': 'Event Types',
     'systemName': 'EventTypes'}}}],
 'duration': 5}


This requests the item that belongs to a certain parameter.

In [11]:
testlink14 = regUrl + "items" + "?where=parameters.id=in=" + "101951" 
a=getContent(testlink14)
a


{'offset': 0,
 'hits': 1,
 'totalHits': 1,
 'records': [{'@uuid': 'fd94c2e3-34bf-41b7-9f1f-1e52be21811a',
   'id': 7672,
   'created': '2024-05-07T06:31:14.878',
   'lastModified': '2025-02-09T16:24:29.114',
   'code': 'ctd:ctd_awi_1420',
   'shortName': 'ctd_awi_1420',
   'longName': 'SST_CTD_1420',
   'description': 'High quality, high accuracy multi parameter probe for oceanographic and limnological measurements of physical, chemical and optical parameters.',
   'model': 'CTD 90 M Series II',
   'manufacturer': 'Sea and Sun Technology',
   'serialNumber': '1420',
   'inventoryNumber': '',
   'citation': '10013/sensor.537c5ab9-e425-4c9b-9265-9382045c9035',
   'status': {'@uuid': 'a080ff68-30b2-450a-9a44-06ecd244c0b7',
    'id': 1,
    'lastModified': '2020-11-18T09:14:53.011',
    'name': 'public',
    'meta': 'Item is completely described and operational.'},
   'type': {'@uuid': 'a9d74710-fd32-4820-b27b-5b01a9286138',
    'id': 122,
    'lastModified': '2020-11-18T09:14:53.258',
   


And in case that more than one parameter is of interest, the parameter ids can be submitted via list

In [12]:
testlink15 = regUrl + "items" + "?where=parameters.id=in=" + "(101951,116497, 97155)" 
a=getContent(testlink15)
a


{'offset': 0,
 'hits': 3,
 'totalHits': 3,
 'records': [{'@uuid': '8af5cb0d-f19c-4d4e-ba2d-dc950682c875',
   'id': 7495,
   'created': '2024-09-30T10:35:44.026',
   'lastModified': '2024-09-30T10:35:44.027',
   'code': 'vessel:sonne:adcp_38khz_650074',
   'shortName': 'ADCP_38kHz_650074',
   'longName': 'Acoustic Doppler Current Profiler Ocean Surveyor 38 kHz',
   'description': 'Acoustic Doppler Current Profiler transducer at a frequency of 38 kHz with a maximum range of 1000 m and a maximum ping rate of 0.4 Hz.',
   'model': 'Ocean Surveyor 38 kHz',
   'manufacturer': 'Teledyne RD Instruments',
   'serialNumber': 'Transducer: 650074, Deck Unit: 41140',
   'inventoryNumber': '',
   'citation': '10013/sensor.f157f995-f782-4567-b60f-42da5cac7e0e',
   'parentId': 7108,
   'status': {'@uuid': 'a080ff68-30b2-450a-9a44-06ecd244c0b7',
    'id': 1,
    'lastModified': '2020-11-18T09:14:53.011',
    'name': 'public',
    'meta': 'Item is completely described and operational.'},
   'type': {'@u


In contrast to `=in=` by using `=out=` the returned items do explicitly do not contain the parameters from the submitted list. 

In [13]:
testlink16 = regUrl + "items" + "?where=parameters.id=out=" + "(101951, 116497, 97155)" 
a=getContent(testlink16)
a

{'offset': 0,
 'hits': 3,
 'totalHits': 3688,
 'records': [{'@uuid': '12bd8519-ee5f-4d5e-86c6-7175e8dab660',
   'id': 22,
   'created': '2016-01-11T00:00:00',
   'lastModified': '2024-01-18T07:34:31.561',
   'code': 'vessel:polarstern:weather',
   'shortName': 'Weather',
   'longName': 'Weather Station',
   'description': 'The weather station at Polarstern consists of several devices which measure standard parameters such as temperature, wind, humidity, barometric pressure, radiation and precipitation. Satellite images are received with a dedicated antenna which, together with the data, are used for forecast purposes. The station is operated by a weather technician-observer from the German Weather Service (DWD) who is responsible for the routine 3-hourly synoptic observations and the daily upper air soundings.\n',
   'model': 'AWI',
   'manufacturer': 'AWI',
   'serialNumber': '',
   'inventoryNumber': '',
   'citation': '10013/sensor.14ff286a-af24-4be9-b751-6624db07ddd0',
   'parentId